**ALESSANDRO BARBOSA DE OLIVEIRA**

# Primeiro multiagente — Livraria

Fluxo em níveis: **cliente → estoque → atendimento → venda → cliente**.

- O **estoque** identifica o livro e chama a ferramenta `verificar_estoque`.
- O **atendimento** recebe o pedido e o parecer do estoque e chama a ferramenta `responder_cliente`.
- A ferramenta `responder_cliente` chama o **agente de venda**, que escreve a resposta ao cliente: se o livro estiver disponível, pergunta se deseja reservar ou comprar; se não, pergunta se deseja outro livro.

In [1]:
# imports
import os, unicodedata
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_ollama import ChatOllama

In [2]:
# modelo local no Ollama (API nativa: sem /v1 no final da URL)
MODEL = 'qwen3:4b'
BASE_URL = 'http://localhost:11434'
modelo = ChatOllama(model=MODEL, base_url=BASE_URL,
    temperature=0, num_predict=256, validate_model_on_init=False)

print(f'Usando o modelo {MODEL} em {BASE_URL}')

Usando o modelo qwen3:4b em http://localhost:11434


In [6]:
# dados do estoque
ESTOQUE = {
    "Ensino Médio": 5,
    "Informática": 1,
    "Concurso": 2,
    "Autoajuda": 0,
}

def normalizar(texto: str) -> str:
    """Remove acentos, espaços e maiúsculas para comparar 'informatica' com 'Informática'."""
    sem_acento = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return sem_acento.strip().lower()

In [ ]:
# ferramentas dos agentes
@tool
def verificar_estoque(livro: str) -> str:
    """Consulta o estoque e informa se o livro está disponível e quantas unidades existem.
    Use sempre que precisar saber a disponibilidade de um livro."""
    for nome, quantidade in ESTOQUE.items():
        if normalizar(nome) == normalizar(livro):
            if quantidade > 0:
                return f"Livro disponível. Quantidade em estoque: {quantidade}"
            break
    return "livro indisponível"

@tool
def responder_cliente(parecer_estoque: str) -> str:
    """Chama o agente de venda para escrever a resposta ao cliente a partir do parecer do estoque.
    Use sempre que tiver o parecer do estoque."""
    mensagem_venda = 'Pedido do cliente: ' + pedido_cliente + '\nParecer do estoque: ' + parecer_estoque
    resposta_venda = agente_venda.invoke({'messages': [{'role': 'user', 'content': mensagem_venda}]},
                                         config={'recursion_limit': 8})
    print(resposta_venda)  # para debug, mostra a resposta do agente de venda no formato json
    return resposta_venda['messages'][-1].content

@tool
def fechar_pedido_cliente(pedido_cliente: str, parecer_estoque: str) -> str:
    """Chama o agente de fechamento de pedidos para realizar o fechamento do pedido a partir do parecer do estoque.
    Use sempre que tiver o parecer do estoque."""
    mensagem_fechamento = 'Pedido do cliente: ' + pedido_cliente + '\nParecer do estoque: ' + parecer_estoque
    resposta_fechamento = agente_fechar_pedidos.invoke({'messages': [{'role': 'user', 'content': mensagem_fechamento}]},
                                                       config={'recursion_limit': 8})
    print(resposta_fechamento)
    return resposta_fechamento['messages'][-1].content

print(verificar_estoque.invoke({'livro': 'informatica'}))
print(verificar_estoque.invoke({'livro': 'ensino medio'}))
print(fechar_pedido_cliente.invoke({'pedido_cliente': 'informatica', 'parecer_estoque': 'Livro disponível. Quantidade em estoque: 5'}))

Livro disponível. Quantidade em estoque: 1
Livro disponível. Quantidade em estoque: 5


In [ ]:
# agentes utilizados no fluxo de atendimento
agente_estoque = create_agent(modelo, tools=[verificar_estoque], system_prompt=(
    'Você é o agente de estoque de uma livraria. Livros do catálogo: Ensino Médio, Informática, Concurso. '
    'Identifique o livro pedido pelo cliente e sempre chame a ferramenta verificar_estoque passando apenas o nome do livro, '
    'mesmo que o livro não esteja no catálogo.'))

agente_venda = create_agent(modelo, tools=[], system_prompt=(
    'Você é o agente de venda de uma livraria. Você recebe o pedido do cliente e o parecer do estoque. '
    'Responda ao cliente em português, em até duas linhas, de forma cordial, '
    'informando a disponibilidade e a quantidade em estoque. Não invente livros nem quantidades. '
    'Se o livro estiver disponível, informe a quantidade e pergunte: Deseja reservar ou comprar o livro? '
    'Se o livro estiver indisponível, pergunte: Deseja escolher outro livro?'))

agente_atendimento = create_agent(modelo, tools=[responder_cliente], system_prompt=(
    'Você é o agente de atendimento de uma livraria. Você recebe o pedido do cliente e o parecer do estoque. '
    'Sempre chame a ferramenta responder_cliente passando o parecer do estoque. '
    'Responda somente com o texto devolvido pela ferramenta, sem acrescentar nada.'))

agente_fechar_pedidos = create_agent(modelo, tools=[fechar_pedido_cliente], system_prompt=(
    "Você é o agente de fechamento de pedidos de uma livraria. Você recebe o pedido do cliente e o parecer do estoque, conforme reserva do cliente."
    "Sempre chame a ferramenta fechar_pedido_cliente passando o pedido do cliente e o parecer do estoque."))


In [10]:
# função para criar o envelope da mensagem do agente de pedidos para o agente de vendas
def criar_envelope(conteudo, remetente, destinatario):
    return{
        "id": "mensagem-001 ",
        "conteudo": conteudo,
        "remetente": remetente,
        "destinatario": agente_venda,
        "acao":'responder_cliente'
    }


In [15]:
# 1. cliente faz o pedido
pedido_cliente = input('Cliente: ')

In [21]:
# 2. agente de estoque consulta a ferramenta verificar_estoque
resposta_estoque = agente_estoque.invoke(
    {'messages': [{'role': 'user', 'content': 'Pedido do cliente: ' + pedido_cliente}]},
    config={'recursion_limit': 8})
print(resposta_estoque) 
ferramentas_estoque = [m for m in resposta_estoque['messages'] if isinstance(m, ToolMessage)]
if ferramentas_estoque:
    parecer_estoque = ferramentas_estoque[-1].content
else:
    parecer_estoque = 'livro indisponível'
print('Estoque → atendimento:', parecer_estoque)

{'messages': [HumanMessage(content='Pedido do cliente: Olá, você tem livros de Informática', additional_kwargs={}, response_metadata={}, id='4dd98837-f551-442a-8aa4-e44cf781443b'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-11-03T19:47:34.2227349Z', 'done': True, 'done_reason': 'length', 'total_duration': 7683003500, 'load_duration': 28798100, 'prompt_eval_count': 245, 'prompt_eval_duration': 789015000, 'eval_count': 256, 'eval_duration': 6834137000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a1a2f3-bad9-7319-a73b-fdd323ecc570-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 245, 'output_tokens': 256, 'total_tokens': 501})]}
Estoque → atendimento: livro indisponível


In [23]:
# 3. atendimento recebe o parecer e chama responder_cliente, que aciona o agente de venda
mensagem_atendimento = 'Pedido do cliente: ' + pedido_cliente + '\nParecer do estoque: ' + parecer_estoque
resposta_atendimento = agente_atendimento.invoke(
    {'messages': [{'role': 'user', 'content': mensagem_atendimento}]},
    config={'recursion_limit': 8})

ferramentas_atendimento = [m for m in resposta_atendimento['messages'] if isinstance(m, ToolMessage)]
resposta_cliente = ferramentas_atendimento   # texto escrito pelo agente de venda
print('Venda → cliente:', resposta_cliente)

Venda → cliente: []
